# Decision Tree

Calculate classification impurity and compare one-level regression-tree splits.

Run the setup cell first. Each numbered section has editable inputs followed by a
calculation cell. Run that section again after changing its inputs, or use
**Restart Kernel and Run All Cells** to run the examples from a clean state.
Run each workflow's calculation before its result-detail cells. Every input has
a comment describing its format or choices. Optional fields accept `None`,
`"None"`, or blank text. Rerun setup after updating these files.

Matrices accept `1 2; 3 4`, rows on separate lines, fractions such as `1/3`,
nested lists, and NumPy arrays. Rows usually represent samples and columns features.
Results shown with `cache=cache` are saved automatically for use in any notebook
through `cache.load("name")`. The [Cache notebook](09_cache.ipynb) manages them.

[Setup and notebook index](README.md)

In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in this folder, the repository root, or a subfolder.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "CommonUtils.py").is_file() and (p / "notebooks" / "notebook_tools.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from the ee2211-solver repository or its notebooks folder.")
for folder in (PROJECT_ROOT, PROJECT_ROOT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

import numpy as np
import pandas as pd
import sympy as sp
from IPython.display import Markdown, display
import importlib
import notebook_tools
importlib.reload(notebook_tools)
from notebook_tools import (NotebookCache, matrix, optional_matrix, is_blank,
    number, integer, boolean, variable_names, coordinates, choice, show, show_metrics)

pd.set_option("display.precision", 10)
cache = NotebookCache()
print(f"Python {sys.version.split()[0]} | shared cache: {cache.path}")

## 1. Classification impurity
Each column is a class. With `includes_parent = True`, the first row is the
parent and remaining rows are disjoint children. Parent counts must equal their
column-wise sums. With `False`, enter only children. Only children contribute to
weighted impurity; unused children have zero weight.

Measure can be `all`, `gini`, `entropy` (base 2), or `misclassification`.

In [ ]:
class_counts = '8 5 5; 6 4 0; 2 1 5'  # Node rows, class-count columns; optional parent first.
parent_first = True  # True = first row is parent; False = child rows only.
measure = 'all'  # all | gini | entropy | misclassification.

In [ ]:
from DecisionTreeUtils import CLASSIFICATION_MEASURES, classification_split_details

counts = matrix(class_counts)
measures = CLASSIFICATION_MEASURES if measure == "all" else (measure,)
classification_summaries = {}
for selected_measure in measures:
    details = classification_split_details(counts, selected_measure, boolean(parent_first))
    classification_summaries[selected_measure] = details
    display(Markdown("### " + selected_measure.title()))
    display(pd.DataFrame({"Child node": np.arange(1, len(details["node_sizes"])+1),
        "Samples": details["node_sizes"], "Weight": details["weights"],
        "Impurity": details["impurities"], "Weighted contribution": details["weighted_contributions"]}))
    show_metrics({key: details[key] for key in ("parent_impurity", "weighted_impurity", "gain")})
    cache.save(selected_measure + "_per_node", details["impurities"].reshape(-1, 1), source="Classification tree")
    cache.save("weighted_" + selected_measure + "_per_node", details["weighted_contributions"].reshape(-1, 1), source="Classification tree")
    cache.save("weighted_" + selected_measure, [[details["weighted_impurity"]]], source="Classification tree")

## 2. Regression MSE and splits
Enter one feature and target per sample. The course convention is **left x < t**,
**right x ≥ t**. Each node predicts its mean; overall MSE weights child MSEs by
sample counts. A threshold must leave at least one sample on each side.

Set `threshold = None` to skip a supplied threshold. With `find_best = True`,
all midpoints between distinct sorted x values are checked; ties choose the first.

In [ ]:
x = '0.2; 0.7; 1.8; 2.2; 3.7; 4.1'  # One numeric feature per sample.
y = '2.1; 1.5; 5.8; 6.1; 9.1; 9.5'  # One target per sample; same count as x.
threshold = 2.0  # Numeric split point; None skips this threshold.
find_best = True  # True searches all candidate midpoints; False skips search.

In [ ]:
from DecisionTreeUtils import regression_threshold_summary, find_best_regression_split

threshold = number(threshold, optional=True)
find_best = boolean(find_best)
tree_x, tree_y = matrix(x), matrix(y)
summary_keys = ("threshold", "root_mean", "root_mse", "left_size", "left_mean", "left_mse", "right_size", "right_mean", "right_mse", "overall_mse")
if threshold is None and not find_best:
    raise ValueError("Supply a threshold or enable find_best.")
if threshold is not None:
    threshold_summary = regression_threshold_summary(tree_x, tree_y, threshold)
    display(Markdown("### Supplied threshold"))
    show_metrics({key: threshold_summary[key] for key in summary_keys})
    cache.save("threshold_summary", [[threshold_summary[key]] for key in ("threshold", "root_mse", "left_mse", "right_mse", "overall_mse")], source="Threshold, root MSE, left MSE, right MSE, split MSE")
if find_best:
    best_split, split_candidates = find_best_regression_split(tree_x, tree_y)
    display(Markdown("### Candidate splits"))
    display(pd.DataFrame([{key: item[key] for key in ("threshold", "overall_mse", "left_mse", "right_mse")} for item in split_candidates]))
    display(Markdown("### Best split"))
    show_metrics({key: best_split[key] for key in summary_keys})
    show("best_split", [[best_split["threshold"]], [best_split["overall_mse"]]], cache=cache, source="Threshold and split MSE")